### This script was initially a copy of testingScript.ipynb. The purpose is to retireve test results for all prompts in order to establish cosine similarity thresholds (e.g. Excellent, Good, Poor) for the model.
This is how the data is obtained for the confidence boxplots. The estimated relevance labels are established using the bge-small-en-v1.5 model (note that as opposed to testingScript.py, this script only uses one embedding model)

#### Setup

In [1]:
import os
print(os.getcwd())
new_directory = "../.." # Example of going up one level and into 'some_other_folder'
# Or simply the name of a subdirectory:
# new_directory = "my_subdirectory"

try:
    # Change the current working directory
    os.chdir(new_directory)

    # Print the current working directory after changing
    print(f"Current working directory (after): {os.getcwd()}")

except FileNotFoundError:
    print(f"Error: The directory '{new_directory}' does not exist.")
except NotADirectoryError:
    print(f"Error: '{new_directory}' is a file, not a directory.")
except PermissionError:
    print(f"Error: You do not have permission to access '{new_directory}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

c:\Users\Tillman\Thesis\testing_files\ConfidenceTesting
Current working directory (after): c:\Users\Tillman\Thesis


In [2]:
import pandas as pd
import numpy as np
import datetime as dt
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from utils import split_into_sentences, merge_session_times
from utils import EmbModel
import time

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
2025-05-19 13:15:20.025 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
[nltk_data] Downloading package punkt_tab to c:\Users\Tillman\anaconda
[nltk_data]     3\envs\ThesisModel\lib\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to c:\U
[nltk_data]     sers\Tillman\anaconda3\envs\ThesisModel\lib\nltk_data.
[nltk_data]     ..
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to c:\Users\Tillman\anaconda3\
[nl

#### Defining new testResult object

In [3]:
class testResult:
    def __init__(self, model_name_short, dataset, prompt, promptType):
        self.model_name_short = model_name_short
        self.dataset = dataset
        self.prompt = prompt
        self.promptType = promptType
        self.time = None
        self.result = None
        self.reranking_score_1 = None
        self.reranking_score_2 = None
        self.retrival_score = None

    def set_result_df(self, result):
        self.result = result

    def set_time(self, time):
        self.time = time  

    def set_reranking_1(self, reranking_score_1):
        self.reranking_score_1 = reranking_score_1

    def set_reranking_1(self, reranking_score_2):
        self.reranking_score_2 = reranking_score_2

    def set_retrieval(self, retrieval_score):
        self.retrieval_score = retrieval_score

#### Data Processing

In [4]:
#Rewriting get_best_abstracts function to take parameters
def get_best_abstracts(embeddings, lem_df, original_df, string, embedding_model):

    def get_best_embedding(string, embedding_model = embedding_model, embeddings = embeddings, lem_df = lem_df):
        str_embedding = embedding_model.encode(string)
        str_embedding_reshaped = str_embedding.reshape(1, -1)  # Reshape only once

        # Vectorized cosine similarity calculation
        cosine_similarities = cosine_similarity(np.array(embeddings), str_embedding_reshaped)
        cosine_similarities = cosine_similarities.flatten()  # Flatten to a 1D array

        lem_df['cosine_similarity'] = cosine_similarities

        test_df = lem_df.groupby(by='paper_id')['cosine_similarity'].mean()

        return test_df
    
#Splitting prompt into sentences
    prompt_df = pd.DataFrame()
    prompt_df['text'] = [string]
    if embeddings.doChunk:
        prompt_df = split_into_sentences(prompt_df)

    #Matching embeddings for each sentence
    agg_df = pd.DataFrame()

    #Splitting into sentences but if the model does not require chunking,
    #Then each 'sentence' will actually be the entire body of text
    for sentence in prompt_df['text']:
        sent_df = get_best_embedding(string=sentence, 
                                     embeddings = embeddings.embedding, 
                                     lem_df = lem_df,
                                     embedding_model=embedding_model)
        sent_df = pd.DataFrame(sent_df)
        sent_df.reset_index(names='paper_id', inplace=True)
        agg_df = pd.concat([agg_df, sent_df])

    #Aggregating
    agg_df = agg_df.groupby(by='paper_id').mean('cosine_similarity')

    #Here you can weigh the different cosine similarities somehow
    #Percentile mean code
    # else:
    #     agg_df = agg_df.groupby(by='cosine_similarity')
    #     agg_df = top_percentile_mean(agg_df, 'cosine_similarity', percentile= 0.6)
    #     agg_df.columns = ['cosine_similarity']

    output_df = agg_df.merge(original_df, left_index=True, right_on = 'paper_id')
    output_df.sort_values(by='cosine_similarity', ascending=False, inplace=True)

    output_df['text'] = output_df['paper_title'] + ". " + output_df['abstract']
    output_df = output_df[['text','cosine_similarity']]
    
    #TODO: output_df should contain cosine_similarity score and concatenated text comprised of paper_title and abstract
    return output_df

In [5]:
def get_result(embedding, pre_embedding_df, presentation_df, 
               model_name_short, emb_model, dataset, prompt_str, prompt_type):
    current_result = testResult(model_name_short=model_name_short,
                                dataset=dataset,
                                prompt=prompt_str,
                                promptType=prompt_type,
                                )

    start_time = time.perf_counter()

    result_df = get_best_abstracts(embeddings=embedding,
                            lem_df=pre_embedding_df,
                            original_df=presentation_df,
                            embedding_model=emb_model, 
                            string=current_result.prompt)

    current_result.set_result_df(result_df.head(10))

    end_time = time.perf_counter()
    total_time = end_time-start_time

    current_result.set_time(total_time)
    return current_result


#### Combined script

In [6]:
dataset_list = ['IISE 2024', 'INFORMS 2024', 'IISE 2025']
model_list = ['BAAI/bge-small-en-v1.5']

result_list = []

for dataset in dataset_list:
    print('----------------------')
    print(f"Dataset: {dataset}")
    dataset_name_nospace = dataset.replace(" ","")
    session_filepath = f'./Datasets/{dataset}/session_times.csv'
    queries_filepath = f'./Datasets/{dataset}/Confidence testing/testing_queries.xlsx'
    presentation_filepath = f'./Datasets/{dataset}/{dataset_name_nospace}_Processed.csv'

    presentation_df = pd.read_csv(presentation_filepath)
    session_df = pd.read_csv(session_filepath)
    queries_df = pd.read_excel(queries_filepath)

    for model in model_list:
        start_time_2 = time.perf_counter()
        print(f"Model: {model}")
        model_name_short = model.split('/')[-1]
        embedding_filepath = f'./Datasets/{dataset}/Embeddings/{model_name_short}.pkl'

        emb = open(embedding_filepath, 'rb') 
        embedding = pickle.load(emb)
        emb.close()
        embedding_model = SentenceTransformer(model)

        if embedding.doChunk:
            pre_emb_path = f'./Datasets/{dataset}/chunked_df.csv'
        else:
            pre_emb_path = f'./Datasets/{dataset}/non_chunked_df.csv'

        pre_embedding_df = pd.read_csv(pre_emb_path)

        for index, row in queries_df.iterrows():
            prompt_str = row['Prompt']
            prompt_type = row['Type']
            result = get_result(embedding=embedding,
                                pre_embedding_df=pre_embedding_df,
                                presentation_df=presentation_df,
                                emb_model = embedding_model,
                                model_name_short=model_name_short,
                                dataset=dataset,
                                prompt_str=prompt_str,
                                prompt_type=prompt_type)
            result_list.append(result)
        end_time_2 = time.perf_counter()
        print(f"Testing time: {end_time_2 - start_time_2}")
        print(' ')


----------------------
Dataset: IISE 2024
Model: BAAI/bge-small-en-v1.5
Testing time: 6.360636800003704
 
----------------------
Dataset: INFORMS 2024
Model: BAAI/bge-small-en-v1.5
Testing time: 6.632125399992219
 
----------------------
Dataset: IISE 2025
Model: BAAI/bge-small-en-v1.5
Testing time: 6.377700100012589
 


### For confidence testing: new block of code needed to transform testResult objects into dataframe format

In [7]:
print(len(result_list))
display(result_list[0].result)

285


,text,cosine_similarity
126,Enhancing supply chain resilience: evaluating ...,0.866329
728,Resilient Suppliers Selection in Pharmaceutica...,0.823123
782,Exploring the Network Characteristics pertinen...,0.821306
498,Simulation Analysis of Risk of Job Tardiness a...,0.805583
49,Advancing Supply Chain Sustainability: Environ...,0.801513
821,Modeling and Analyzing Project Resilience Unde...,0.797996
176,Case Study; A Practitioners Guide to Implement...,0.787145
105,Energy Security in Remote Communities: A Resil...,0.786685
554,Fulfilling Triple Bottom Line Objectives: A Ma...,0.784119
670,"A Review of the Performance Indicators, Key Fa...",0.777888


In [8]:
def convert_results_to_dataframe(results_list):
    """
    Converts a 1-D list of testResult objects into a single pandas DataFrame.

    Each testResult object's 'result' DataFrame is used as a base.
    The following columns are added/modified:
    - 'resultOrder': Position based on descending 'cosine_similarity', starting at 1.
    - 'queryType': Value from testResult.promptType.
    - 'dataset': Value from testResult.dataset.

    Args:
        results_list (list): A list of testResult objects.

    Returns:
        pandas.DataFrame: A single DataFrame containing all processed results.
                          Returns an empty DataFrame if results_list is empty or
                          contains no valid result DataFrames.
    """
    processed_dfs = []

    if not results_list:
        return pd.DataFrame() # Return empty DataFrame if input list is empty

    for test_res_obj in results_list:
        # Ensure the result attribute is a DataFrame and not None
        if test_res_obj.result is None or not isinstance(test_res_obj.result, pd.DataFrame):
            print(f"Skipping an object as its 'result' attribute is None or not a DataFrame. Model: {test_res_obj.model_name_short}, Dataset: {test_res_obj.dataset}")
            continue
        
        if test_res_obj.result.empty:
            print(f"Skipping an object as its 'result' DataFrame is empty. Model: {test_res_obj.model_name_short}, Dataset: {test_res_obj.dataset}")
            continue

        # Make a copy to avoid modifying the original DataFrame in the object, if needed
        current_df = test_res_obj.result.copy()

        # Ensure 'cosine_similarity' column exists
        if 'cosine_similarity' not in current_df.columns:
            print(f"Skipping a DataFrame for model {test_res_obj.model_name_short} as 'cosine_similarity' column is missing.")
            continue

        # Sort by 'cosine_similarity' in descending order
        current_df = current_df.sort_values(by='cosine_similarity', ascending=False)

        # Add 'resultOrder' column (position starting from 1)
        # .reset_index(drop=True) is important to ensure the index is 0, 1, 2... for proper ordering
        current_df['resultOrder'] = current_df.reset_index(drop=True).index + 1
        
        # Add 'queryType' column
        current_df['queryType'] = test_res_obj.promptType
        
        # Add 'dataset' column
        current_df['dataset'] = test_res_obj.dataset
        
        processed_dfs.append(current_df)

    if not processed_dfs:
        return pd.DataFrame() # Return empty DataFrame if no valid DataFrames were processed

    # Concatenate all processed DataFrames
    final_df = pd.concat(processed_dfs, ignore_index=True)
    
    return final_df

result_df = convert_results_to_dataframe(results_list=result_list)
result_df = result_df.drop('text', axis=1)
display(result_df)

,cosine_similarity,resultOrder,queryType,dataset
0,0.866329,1,Domain Relevant,IISE 2024
1,0.823123,2,Domain Relevant,IISE 2024
2,0.821306,3,Domain Relevant,IISE 2024
3,0.805583,4,Domain Relevant,IISE 2024
4,0.801513,5,Domain Relevant,IISE 2024
...,...,...,...,...
2845,0.586690,6,Irrelevant,IISE 2025
2846,0.585969,7,Irrelevant,IISE 2025
2847,0.585351,8,Irrelevant,IISE 2025
2848,0.584376,9,Irrelevant,IISE 2025


In [9]:
filename = r'.\r_code\confidence_data.csv'
result_df.to_csv(filename)